# 10年定着予測 - 転居許容×希望勤務地マッチの交互作用（EDA v5由来、18_ベースライン単体検証）

**背景**: `notebooks/eda/data_exploration_v5.ipynb`で、ブロックE（転居許容フラグ）とブロックJ
（希望勤務地マッチ度、`25_`でPublic確認済み・18_ベースライン比+1.76%改善）は、それぞれ単体では
既に有効と確認済みだが、**両者の交互作用（AND条件）は未検証**だった。

「転居を伴う異動を許容しない」**かつ**「希望と異なる勤務地に配属された」社員（n=342, 12.4%）の
定着率はわずか**23.1%**で、それ以外（61.4%）との差は**38.3%pt**（p=1.4×10⁻³⁹、オッズ比0.189）。
これはJ単体（25.3%pt, p=1.4×10⁻²⁶）を上回る、本プロジェクトでこれまで確認された全単一特徴量中で
最大の効果量・有意性である。単純な加法効果ではなく、転居許容者では勤務地不一致の影響が小さい
（-7.2%pt）のに対し、転居非許容者では劇的に大きい（-38.5%pt）という真の交互作用であり、
E・Jの成分がそれぞれ既にモデルに入っていても、決定木がこのAND関係を自動的に効率よく学習できる
とは限らない（詳細は`notebooks/eda/report/md/data_exploration_v5_report.md`セクション6参照）。

## なぜE_memoに追加せず、18_のベースラインに対して単体で検証するのか

`20_`〜`24_`で、Publicで唯一確実に改善したブロックE（メモ構造化）に対しF・G・Iを追加する実験を
3回行ったが、検証では改善または中立に見えてもPublicでは3回連続悪化した
（`data/output/submit_result_report.md`セクション23-31参照）。この教訓を踏まえ、Eを発見したときと
同じ手順（J確立時の`25_`と同じ手順）に立ち返り、新ブロックを**18_のベースライン（Eなし、
D_expanded + TF-IDF A_v1のみ）に対して単体で追加**して検証する。

## 新規ブロック

- **L（転居×勤務地マッチ ダブル悪条件）**: 「転居を伴う異動を許容しない」かつ「希望勤務地と実際の
  配属地が不一致」の交互作用フラグ（+参考用の4値カテゴリ）
- （参考・再現確認用）**J（希望勤務地マッチ度）**: `25_`で確立済み、Public 0.540648

## 検証方法

CatBoost + Optuna（探索範囲は`18_`〜`26_`と同一のn_trials=25）、CPU実行、80/20・75/25の2つの
時系列splitで評価する。baseline・L単体・J単体（再現確認）・combo_JL（参考、即採用はしない）を比較する。

## 実行環境
Google Colab（CPU、ハイメモリ推奨）を想定。


In [4]:
!pip install -q catboost optuna

In [5]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [6]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [7]:
import datetime
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [8]:
SCRIPT_NAME = "27_relocation_location_mismatch_interaction"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-11 04:53:06] [INFO] === [27_relocation_location_mismatch_interaction] 実験開始 ===


INFO:27_relocation_location_mismatch_interaction:=== [27_relocation_location_mismatch_interaction] 実験開始 ===


[2026-08-11 04:53:06] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


INFO:27_relocation_location_mismatch_interaction:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


[2026-08-11 04:53:06] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/27_relocation_location_mismatch_interaction_checkpoint.csv


INFO:27_relocation_location_mismatch_interaction:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/27_relocation_location_mismatch_interaction_checkpoint.csv


[2026-08-11 04:53:06] [INFO] チェックポイントは未作成（新規実行）


INFO:27_relocation_location_mismatch_interaction:チェックポイントは未作成（新規実行）


In [9]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-11 04:53:12] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:27_relocation_location_mismatch_interaction:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-11 04:53:12] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:27_relocation_location_mismatch_interaction:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-11 04:53:12] [INFO] 定着率: 0.5647


INFO:27_relocation_location_mismatch_interaction:定着率: 0.5647


[2026-08-11 04:53:12] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:27_relocation_location_mismatch_interaction:Train IDs: 2761, Test IDs: 2502


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [10]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [11]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-11 04:53:12] [INFO] ------------------------------------------------------------


INFO:27_relocation_location_mismatch_interaction:------------------------------------------------------------


[2026-08-11 04:53:12] [INFO] split非依存の基本特徴量を生成中...


INFO:27_relocation_location_mismatch_interaction:split非依存の基本特徴量を生成中...


[2026-08-11 04:53:12] [INFO] ------------------------------------------------------------


INFO:27_relocation_location_mismatch_interaction:------------------------------------------------------------


[2026-08-11 05:01:04] [INFO] split非依存の基本特徴量生成完了


INFO:27_relocation_location_mismatch_interaction:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [12]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-11 05:01:04] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:27_relocation_location_mismatch_interaction:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-11 05:01:06] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:27_relocation_location_mismatch_interaction:入社時メモ: SVD累積寄与率=0.760


[2026-08-11 05:01:11] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:27_relocation_location_mismatch_interaction:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-11 05:01:14] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:27_relocation_location_mismatch_interaction:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-11 05:01:14] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:27_relocation_location_mismatch_interaction:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [13]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-11 05:01:14] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:27_relocation_location_mismatch_interaction:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-11 05:04:16] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:27_relocation_location_mismatch_interaction:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [14]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-11 05:04:16] [INFO] Persona単位の基本特徴量を生成中...


INFO:27_relocation_location_mismatch_interaction:Persona単位の基本特徴量を生成中...


[2026-08-11 05:04:16] [INFO] Persona単位の基本特徴量処理完了


INFO:27_relocation_location_mismatch_interaction:Persona単位の基本特徴量処理完了


## 5. 希望勤務地マッチ度特徴量（ブロックJ、`25_`で確立済み・参考用に再現）

`25_`でPublic確認済み（Public 0.540648、18_ベースライン比+1.76%改善、検証-Publicギャップ0.0095と健全）。
今回はLとの比較・combo_JLの参考検証のために同一ロジックで再現する。

In [15]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


def extract_desired_location(s):
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def create_location_match_features(persona_df):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    desired = ws_section.apply(extract_desired_location)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()
    match_cat = pd.Series("unknown", index=persona_df.index)
    match_cat[desired.notna()] = match[desired.notna()].map({True: "match", False: "mismatch"})
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "勤務地希望マッチ": match_cat.values,
    })

logger.info("希望勤務地マッチ度特徴量(ブロックJ、参考再現)を生成中...")
train_locmatch = create_location_match_features(train_persona)
test_locmatch = create_location_match_features(test_persona)
logger.info(f"ブロックJ: Train {train_locmatch.shape}, Test {test_locmatch.shape}")
train_locmatch["勤務地希望マッチ"].value_counts()

[2026-08-11 05:04:16] [INFO] 希望勤務地マッチ度特徴量(ブロックJ、参考再現)を生成中...


INFO:27_relocation_location_mismatch_interaction:希望勤務地マッチ度特徴量(ブロックJ、参考再現)を生成中...


[2026-08-11 05:04:16] [INFO] ブロックJ: Train (2761, 2), Test (2502, 2)


INFO:27_relocation_location_mismatch_interaction:ブロックJ: Train (2761, 2), Test (2502, 2)


,count
勤務地希望マッチ,
match,1863
mismatch,584
unknown,314


## 6. 転居許容×勤務地マッチの交互作用特徴量（ブロックL、EDA v5由来・新規）

`入社時メモ`の「勤務地・働き方」セクションから「転居許容」フラグ（`data_exploration_v3.ipynb`と
同一ロジック）と「希望勤務地」（ブロックJと同一ロジック）を抽出し、両者の交互作用を特徴量化する。
**EDA v5で確認済みの本プロジェクト最大の効果量（差38.3%pt, p=1.4×10⁻³⁹）**を持つ。

In [16]:
NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def create_relocation_mismatch_features(persona_df):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_desired_location)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "転居x勤務地_状態": state.values,
        "転居x勤務地_ダブル悪条件": double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL)を生成中...")
train_relocmismatch = create_relocation_mismatch_features(train_persona)
test_relocmismatch = create_relocation_mismatch_features(test_persona)
logger.info(f"ブロックL: Train {train_relocmismatch.shape}, Test {test_relocmismatch.shape}")
print(train_relocmismatch["転居x勤務地_状態"].value_counts())
print()
print(train_relocmismatch["転居x勤務地_ダブル悪条件"].value_counts())

[2026-08-11 05:04:16] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL)を生成中...


INFO:27_relocation_location_mismatch_interaction:転居×勤務地マッチ交互作用特徴量(ブロックL)を生成中...


[2026-08-11 05:04:16] [INFO] ブロックL: Train (2761, 3), Test (2502, 3)


INFO:27_relocation_location_mismatch_interaction:ブロックL: Train (2761, 3), Test (2502, 3)


転居x勤務地_状態
非許容_一致     1072
許容_一致       776
非許容_不一致     342
unknown     336
許容_不一致      235
Name: count, dtype: int64

転居x勤務地_ダブル悪条件
0    2419
1     342
Name: count, dtype: int64


## 7. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"J"}`/`{"L"}`/`{"J","L"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してブロックを単体・組み合わせで追加できるようにする。

In [17]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None):
    '''指定した分割比率で特徴量を組み立てる。extra_blocks: {"J","L"}のサブセット'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "J" in extra_blocks:
        tf = tf.merge(train_locmatch, on=ID_COL, how="left")
        ttf = ttf.merge(test_locmatch, on=ID_COL, how="left")

    if "L" in extra_blocks:
        tf = tf.merge(train_relocmismatch, on=ID_COL, how="left")
        ttf = ttf.merge(test_relocmismatch, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

✅ 部署Target Encoding・prepare_split関数定義完了


## 8. チェックポイント機能（`18_`〜`26_`と同一）

In [18]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

✅ チェックポイント関数定義完了


## 9. モデル実行関数（CatBoost、継続検証中のモデル）

`18_`のステップBでCatBoostが大差で最良だったため、本ノートブックではCatBoostのみで検証する。

In [19]:
def run_model_config(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]
    X_test = test_features[feature_cols].fillna(-999)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    val_preds = final_model.predict_proba(X_va)[:, 1]
    test_preds = final_model.predict_proba(X_test)[:, 1]

    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)

    logger.info(f"[{config_label}] n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {
        "config": config_label, "n_features": len(feature_cols), "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }

print("✅ run_model_config関数定義完了")

✅ run_model_config関数定義完了


## 10. アブレーション: baseline(18_、Eなし) / L単体 / J単体(参考再現) / combo_JL(参考) × 2 split

baseline（extra_blocks=なし、= `18_`のCatBoost+D_expanded相当、**Eは含まない**）に対して
**L（転居×勤務地マッチ ダブル悪条件、EDA v5最大の発見）を最優先で検証**する。J単体は`25_`の
Public確認済み結果（0.540648）との再現性チェックを兼ねる参考。combo_JLは組み合わせの参考値であり、
`20_`〜`26_`の教訓（検証で改善に見えてもPublicで悪化するケースが複数あった）を踏まえ、
検証結果だけでは即採用しない。

In [20]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
BLOCK_CONFIGS = {
    "baseline": set(),
    "L_reloc_mismatch": {"L"},
    "J_location_match": {"J"},
    "combo_JL": {"J", "L"},
}

ablation_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for block_name, blocks in BLOCK_CONFIGS.items():
        config_label = f"{split_name}_{block_name}"
        def _run(ratio=ratio, blocks=blocks, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, extra_blocks=blocks)
            return run_model_config(ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25)
        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["block_config"] = block_name
        ablation_results.append(result)

ablation_df = pd.DataFrame(ablation_results)
ablation_pivot = ablation_df.pivot(index="block_config", columns="split", values="val_score")
ablation_pivot["mean"] = ablation_pivot[["split_80_20", "split_75_25"]].mean(axis=1)
ablation_pivot["std"] = ablation_pivot[["split_80_20", "split_75_25"]].std(axis=1)
ablation_pivot = ablation_pivot.reindex(["baseline", "L_reloc_mismatch", "J_location_match", "combo_JL"])
ablation_pivot["mean_diff_vs_baseline"] = ablation_pivot["mean"] - ablation_pivot.loc["baseline", "mean"]
ablation_pivot = ablation_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("アブレーション結果（baseline vs L/J単体 vs combo_JL）")
logger.info("=" * 60)
logger.info("\n" + ablation_pivot.to_string())
print("\n■ アブレーション結果:")
print(ablation_pivot.to_string())
print("\n※ CPU実行のため、GPU実測値(18_やE_memo等)とは直接比較しない。baselineとの差分のみで判断する")
print("※ J_location_matchはPublic 0.540648で確認済み。今回の値との整合性チェックにも使う")

[2026-08-11 05:04:17] [INFO] === split_80_20_baseline ===


INFO:27_relocation_location_mismatch_interaction:=== split_80_20_baseline ===


[2026-08-11 05:09:19] [INFO] [split_80_20_baseline] n_features=439, val_score=0.554220


INFO:27_relocation_location_mismatch_interaction:[split_80_20_baseline] n_features=439, val_score=0.554220


[2026-08-11 05:09:19] [INFO] === split_80_20_L_reloc_mismatch ===


...

[2026-08-11 05:37:45] [INFO] ============================================================


INFO:27_relocation_location_mismatch_interaction:============================================================


[2026-08-11 05:37:45] [INFO] 
split             split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
block_config                                                                         
combo_JL             0.522475     0.507339  0.514907  0.010703              -0.039124
L_reloc_mismatch     0.521117     0.509400  0.515258  0.008286              -0.038772
J_location_match     0.537880     0.531138  0.534509  0.004767              -0.019522
baseline             0.553842     0.554220  0.554031  0.000267               0.000000


INFO:27_relocation_location_mismatch_interaction:
split             split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
block_config                                                                         
combo_JL             0.522475     0.507339  0.514907  0.010703              -0.039124
L_reloc_mismatch     0.521117     0.509400  0.515258  0.008286              -0.038772
J_location_match     0.537880     0.531138  0.534509  0.004767              -0.019522
baseline             0.553842     0.554220  0.554031  0.000267               0.000000



■ アブレーション結果:
split             split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
block_config                                                                         
combo_JL             0.522475     0.507339  0.514907  0.010703              -0.039124
L_reloc_mismatch     0.521117     0.509400  0.515258  0.008286              -0.038772
J_location_match     0.537880     0.531138  0.534509  0.004767              -0.019522
baseline             0.553842     0.554220  0.554031  0.000267               0.000000

※ CPU実行のため、GPU実測値(18_やE_memo等)とは直接比較しない。baselineとの差分のみで判断する
※ J_location_matchはPublic 0.540648で確認済み。今回の値との整合性チェックにも使う


## 11. 総合結果・提出候補

split_80_20における各構成の提出ファイルパスを一覧化する。**L単体が最も優先度の高い提出候補**。
combo_JLが単体を上回っていても、`20_`〜`26_`の教訓を踏まえて即採用せず、両splitでの一貫性を
確認した上で、必ずPublicで単体から確認する。

In [21]:
split_80_20_rows = ablation_df[ablation_df["split"] == "split_80_20"].set_index("block_config")
summary_rows = split_80_20_rows[["val_score", "submission_path"]].reindex(
    ["baseline", "L_reloc_mismatch", "J_location_match", "combo_JL"]
).reset_index()

logger.info("=" * 60)
logger.info("総合結果（split_80_20、提出候補一覧）")
logger.info("=" * 60)
logger.info("\n" + summary_rows.to_string())
print("\n■ 総合結果（split_80_20、提出候補一覧）:")
print(summary_rows.to_string(index=False))
print(f"\n(参考) 20_ E_memo単体: Public 0.534829（現時点の最良）")
print(f"(参考) 25_ J_location_match単体: Public 0.540648")
print(f"(参考) 18_ CatBoost+D_expanded: Public 0.550352")

summary_rows

[2026-08-11 05:37:45] [INFO] ============================================================


INFO:27_relocation_location_mismatch_interaction:============================================================


[2026-08-11 05:37:45] [INFO] 総合結果（split_80_20、提出候補一覧）


INFO:27_relocation_location_mismatch_interaction:総合結果（split_80_20、提出候補一覧）


[2026-08-11 05:37:45] [INFO] ============================================================


INFO:27_relocation_location_mismatch_interaction:============================================================


[2026-08-11 05:37:45] [INFO] 
       block_config  val_score                                                                                                                                submission_path
0          baseline   0.554220          /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_baseline.csv
1  L_reloc_mismatch   0.509400  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_L_reloc_mismatch.csv
2  J_location_match   0.531138  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_J_location_match.csv
3          combo_JL   0.507339          /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_combo_JL.csv


INFO:27_relocation_location_mismatch_interaction:
       block_config  val_score                                                                                                                                submission_path
0          baseline   0.554220          /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_baseline.csv
1  L_reloc_mismatch   0.509400  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_L_reloc_mismatch.csv
2  J_location_match   0.531138  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_J_location_match.csv
3          combo_JL   0.507339          /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_combo_JL.csv



■ 総合結果（split_80_20、提出候補一覧）:
    block_config  val_score                                                                                                                               submission_path
        baseline   0.554220         /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_baseline.csv
L_reloc_mismatch   0.509400 /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_L_reloc_mismatch.csv
J_location_match   0.531138 /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_J_location_match.csv
        combo_JL   0.507339         /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_27_relocation_location_mismatch_interaction_split_80_20_combo_JL.csv

(参考) 20_ E_memo単体: Public 0.534829（現時点の最良）
(参考) 25_ J_location_match単体: Public 0.540648
(参考) 18_ CatBoost+D_expanded: Pu

,block_config,val_score,submission_path
0,baseline,0.554220,/content/drive/MyDrive/jaggle_2026/data/output...
1,L_reloc_mismatch,0.509400,/content/drive/MyDrive/jaggle_2026/data/output...
2,J_location_match,0.531138,/content/drive/MyDrive/jaggle_2026/data/output...
3,combo_JL,0.507339,/content/drive/MyDrive/jaggle_2026/data/output...


## 12. まとめ・次のアクション

1. アブレーション結果の表（10節）で、L単体がbaselineに対し明確な改善を示したか確認する。
   EDA v5での効果量（差38.3%pt, p=1.4×10⁻³⁹）はJ（25.3%pt）を上回るため、L単体はJ単体
   （Public 0.540648）を超える改善が期待できる。
2. **L単体の`split_80_20`提出ファイルを最優先でKaggleに提出**し、Publicスコアを確認する
   （現時点の最良である20_ E_memo単体 Public 0.534829と比較する）。
3. J_location_matchの値が`25_`のPublic確認済み結果（0.540648）と整合的か確認し、パイプラインの
   再現性をチェックする。
4. combo_JLが単体より改善して見えても、`20_`〜`26_`の教訓（組み合わせが検証で改善してもPublicで
   悪化するケースが複数あった）を踏まえ、L単体・J単体それぞれのPublic結果を確認してから
   慎重に判断する。今回はcombo_JLの提出は見送る。
5. もしL単体がE_memoを上回った場合、その新ベースラインに対してE（メモ構造化）を改めて
   追加できないか検討する余地がある（今回は着手しない）。
6. 結果が出たら`data/output/submit_result_report.md`に追記し、メモリ（`best_submission_status.md`・
   `eda_v5_findings.md`）も更新する。

### バックログ（今回は着手しない）
- EDA v5で否定された5候補（部署ID・学歴年齢整合性・勤務地内偏差・前職職種マッチ・志向トラック整合性）は不採用のまま
- テキスト特徴量を日本語の事前学習済み文埋め込みモデルに置き換える案（`19_`で一度試して不採用、将来再検討の余地）
